In [1]:
import os
import cv2
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import GradScaler, autocast
from pathlib import Path
import timm
import albumentations as A
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from tqdm import tqdm

In [2]:
# 2. 경로 설정 (본인의 zip 파일 경로로 수정하세요)
zip_path = "/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/dacon/no_seg&seg_data.zip"
target_path = "/content/no_seg&seg_data"

# 3. /content 폴더로 복사 (드라이브에서 직접 푸는 것보다 복사 후 푸는 게 빠름)
!cp "{zip_path}" /content/

# 4. 압축 해제 (-q 옵션으로 로그 출력 방지)
zip_name = os.path.basename(zip_path)
!unzip -q "/content/{zip_name}" -d "{target_path}"

print(f"✅ {target_path}에 압축 해제 완료!")

✅ /content/no_seg&seg_data에 압축 해제 완료!


In [3]:
# ==========================================
# 1. 설정값 (A100 최적화 및 경로)
# ==========================================
CFG = {
    'seed': 42,
    'img_size': 384,
    'batch_size': 16,
    'epochs': 20,
    'lr': 2e-4,
    'n_folds': 5,
    'model_name': 'convnext_base.fb_in22k_ft_in1k_384',
    'orig_root': Path("/content/no_seg&seg_data/no_seg_data"),
    'mask_root': Path("/content/no_seg&seg_data/seg_data"),
    'train_csv': '/content/no_seg&seg_data/no_seg_data/train.csv',
    'dev_csv': '/content/no_seg&seg_data/no_seg_data/dev.csv'
}

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
# 2. 데이터 통합 및 '이중 층화' 키 생성
train_df = pd.read_csv(CFG['train_csv'])
dev_df = pd.read_csv(CFG['dev_csv'])

train_df['folder'] = 'train'
dev_df['folder'] = 'dev'

all_df = pd.concat([train_df, dev_df], axis=0).reset_index(drop=True)
all_df['label_int'] = all_df['label'].map({'unstable': 0, 'stable': 1})

# [핵심] 레이블과 출처를 합친 키 생성 (예: 'stable_train', 'unstable_dev' 등)
all_df['stratify_key'] = all_df['label'].astype(str) + "_" + all_df['folder']

In [5]:
# 3. Dataset (에러 핸들링 강화)
class StabilityDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self): return len(self.df)

    def _load_img_4ch(self, img_id, view, folder):
        orig_path = CFG['orig_root'] / folder / view / f"{img_id}.png"
        mask_path = CFG['mask_root'] / folder / view / f"{img_id}.png"

        orig_img = cv2.imread(str(orig_path))
        if orig_img is None: raise FileNotFoundError(f"❌ 원본 없음: {orig_path}")
        orig_img = cv2.resize(cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB), (CFG['img_size'], CFG['img_size']))

        mask_img = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask_img is None:
            mask_img = np.ones((CFG['img_size'], CFG['img_size']), dtype=np.uint8) * 255
        else:
            mask_img = cv2.resize(mask_img, (CFG['img_size'], CFG['img_size']))

        _, mask_bin = cv2.threshold(mask_img, 10, 1, cv2.THRESH_BINARY)
        img_4ch = np.concatenate([orig_img, mask_bin[..., np.newaxis]], axis=-1)
        return torch.tensor(img_4ch.transpose(2, 0, 1), dtype=torch.float32) / 255.0

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        f_img = self._load_img_4ch(row['id'], 'front', row['folder'])
        t_img = self._load_img_4ch(row['id'], 'top', row['folder'])
        return f_img, t_img, torch.tensor(row['label_int'], dtype=torch.long)

In [6]:
# 4. 모델 구조
class UltimateFusionNet(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, in_chans=4, num_classes=0)
        feat_dim = self.backbone.num_features
        self.gate = nn.Sequential(nn.Linear(feat_dim*2, feat_dim), nn.ReLU(), nn.Linear(feat_dim, 2), nn.Softmax(dim=1))
        self.head = nn.Sequential(nn.Linear(feat_dim, 512), nn.BatchNorm1d(512), nn.GELU(), nn.Dropout(0.3), nn.Linear(512, 2))

    def forward(self, front, top):
        f_f, t_f = self.backbone(front), self.backbone(top)
        g = self.gate(torch.cat([f_f, t_f], dim=1))
        fused = g[:, 0:1] * f_f + g[:, 1:2] * t_f
        return self.head(fused)

In [7]:
# 5. K-Fold 학습 (stratify_key 기준)
skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])

# stratify_key를 사용하여 dev 데이터가 모든 fold에 균등하게 들어가도록 함
for fold, (t_idx, v_idx) in enumerate(skf.split(all_df, all_df['stratify_key'])):
    print(f"\n🚀 Fold {fold+1} 시작 (Dev 데이터 균등 분포 완료)")
    train_fold, val_fold = all_df.iloc[t_idx], all_df.iloc[v_idx]

    train_loader = DataLoader(StabilityDataset(train_fold), batch_size=CFG['batch_size'], shuffle=True, num_workers=4)
    val_loader = DataLoader(StabilityDataset(val_fold), batch_size=CFG['batch_size'], shuffle=False, num_workers=4)

    model = UltimateFusionNet(CFG['model_name']).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.0)
    optimizer = AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=CFG['epochs'])
    scaler = GradScaler('cuda')

    best_val_logloss = float('inf')

    for epoch in range(CFG['epochs']):
        model.train()
        train_loss = 0
        for f, t, l in tqdm(train_loader, desc=f"Fold {fold+1} Ep {epoch+1}"):
            f, t, l = f.to(device), t.to(device), l.to(device)
            optimizer.zero_grad()
            with autocast('cuda'):
                logits = model(f, t)
                loss = criterion(logits, l)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            train_loss += loss.item()

        # [검증 및 LogLoss 계산]
        model.eval()
        val_probs, val_labels = [], []
        with torch.no_grad():
            for f, t, l in val_loader:
                f, t = f.to(device), t.to(device)
                logits = model(f, t)
                probs = F.softmax(logits, dim=1)
                val_probs.extend(probs.cpu().numpy())
                val_labels.extend(l.numpy())

        epoch_logloss = log_loss(val_labels, val_probs, labels=[0, 1])
        scheduler.step()

        print(f"Fold {fold+1} | Ep {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f} | Val LogLoss: {epoch_logloss:.6f}")

        if epoch_logloss < best_val_logloss:
            best_val_logloss = epoch_logloss
            torch.save(model.state_dict(), f'best_model_fold{fold+1}.pth')
            print(f"🌟 Fold {fold+1} 최고기록! (LogLoss: {epoch_logloss:.6f})")

print("\n🎉 모든 폴드 학습 완료! 이제 진짜 0.01 뚫으러 가시죠.")


🚀 Fold 1 시작 (Dev 데이터 균등 분포 완료)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/354M [00:00<?, ?B/s]

Fold 1 Ep 1: 100%|██████████| 55/55 [00:26<00:00,  2.08it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


Fold 1 | Ep 1 | Train Loss: 0.2967 | Val LogLoss: 0.133196
🌟 Fold 1 최고기록! (LogLoss: 0.133196)


...

Fold 5 Ep 19: 100%|██████████| 55/55 [00:12<00:00,  4.53it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


Fold 5 | Ep 19 | Train Loss: 0.0010 | Val LogLoss: 0.000253


Fold 5 Ep 20: 100%|██████████| 55/55 [00:12<00:00,  4.55it/s]


Fold 5 | Ep 20 | Train Loss: 0.0004 | Val LogLoss: 0.000159

🎉 모든 폴드 학습 완료! 이제 진짜 0.01 뚫으러 가시죠.


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


In [8]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np
import cv2
from pathlib import Path
from tqdm import tqdm

# ==========================================
# 1. 추론용 데이터셋 설정
# ==========================================
class StabilityTestDataset(Dataset):
    def __init__(self, df, CFG):
        self.df = df
        self.CFG = CFG

    def __len__(self): return len(self.df)

    def _load_img_4ch(self, img_id, view):
        # 테스트 이미지 경로 설정
        orig_path = self.CFG['orig_root'] / 'test' / view / f"{img_id}.png"
        mask_path = self.CFG['mask_root'] / 'test' / view / f"{img_id}.png"

        # 원본 로드
        orig_img = cv2.imread(str(orig_path))
        if orig_img is None:
            # 파일이 없을 경우 검은색 더미 이미지 생성 (예외 방지)
            orig_img = np.zeros((self.CFG['img_size'], self.CFG['img_size'], 3), dtype=np.uint8)
        else:
            orig_img = cv2.resize(cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB), (self.CFG['img_size'], self.CFG['img_size']))

        # 마스크 로드
        mask_img = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask_img is None:
            mask_img = np.ones((self.CFG['img_size'], self.CFG['img_size']), dtype=np.uint8) * 255
        else:
            mask_img = cv2.resize(mask_img, (self.CFG['img_size'], self.CFG['img_size']))

        _, mask_bin = cv2.threshold(mask_img, 10, 1, cv2.THRESH_BINARY)
        img_4ch = np.concatenate([orig_img, mask_bin[..., np.newaxis]], axis=-1)

        return torch.tensor(img_4ch.transpose(2, 0, 1), dtype=torch.float32) / 255.0

    def __getitem__(self, idx):
        img_id = self.df.iloc[idx]['id']
        f_img = self._load_img_4ch(img_id, 'front')
        t_img = self._load_img_4ch(img_id, 'top')
        return f_img, t_img

# ==========================================
# 2. 앙상블 추론 실행
# ==========================================
# 제공해주신 샘플 양식 로드
submit = pd.read_csv('/content/no_seg&seg_data/no_seg_data/sample_submission.csv')

test_ds = StabilityTestDataset(submit, CFG)
test_loader = DataLoader(test_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=4)

# 5개 폴드 모델 가중치 로드
models = []
for i in range(1, 6):
    model = UltimateFusionNet(CFG['model_name']).to(device)
    model.load_state_dict(torch.load(f'best_model_fold{i}.pth'))
    model.eval()
    models.append(model)

print(f"✅ 5개 폴드 모델 로드 완료. {len(submit)}개의 테스트 데이터 추론을 시작합니다.")

ensemble_probs = []

with torch.no_grad():
    for f, t in tqdm(test_loader, desc="Final Inference"):
        f, t = f.to(device), t.to(device)

        # 배치별로 5개 모델의 확률값 계산
        batch_probs = []
        for model in models:
            logits = model(f, t)
            probs = F.softmax(logits, dim=1) # (B, 2)
            batch_probs.append(probs.cpu().numpy())

        # 5개 모델 결과의 산술 평균
        avg_prob = np.mean(batch_probs, axis=0) # (B, 2)
        ensemble_probs.extend(avg_prob)

ensemble_probs = np.array(ensemble_probs)

# ==========================================
# 3. 샘플 양식에 맞춰 결과 주입 및 저장
# ==========================================
# 학습 시 설정: 0번 인덱스 = unstable, 1번 인덱스 = stable
submit['unstable_prob'] = ensemble_probs[:, 0]
submit['stable_prob'] = ensemble_probs[:, 1]

# 최종 결과 저장
output_name = 'submission_final_ensemble.csv'
submit.to_csv(output_name, index=False)

print(f"\n🚀 최종 제출 파일 생성 완료: {output_name}")
print(submit.head())

✅ 5개 폴드 모델 로드 완료. 1000개의 테스트 데이터 추론을 시작합니다.


Final Inference: 100%|██████████| 63/63 [01:08<00:00,  1.10s/it]


🚀 최종 제출 파일 생성 완료: submission_final_ensemble.csv
          id  unstable_prob  stable_prob
0  TEST_0001       0.000531     0.999470
1  TEST_0002       0.999834     0.000166
2  TEST_0003       0.999882     0.000118
3  TEST_0004       0.999773     0.000227
4  TEST_0005       0.000945     0.999055


In [15]:
import shutil
drive_folder = "/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/dacon/가중치/5-fold-v2"

# 폴더가 없으면 생성
if not os.path.exists(drive_folder):
    os.makedirs(drive_folder)
    print(f"📂 새 폴더를 생성했습니다: {drive_folder}")

# 3. 파일 복사 (파일명에 실험 정보 추가)
source_file = "submission_final_ensemble.csv"
destination_file = os.path.join(drive_folder, "submission_final_ensemble_v2.csv")

if os.path.exists(source_file):
    shutil.copy2(source_file, destination_file)
    print(f"✅ 복사 완료: {destination_file}")
else:
    print("❌ 복사할 파일(best_model.pth)을 찾을 수 없습니다.")

✅ 복사 완료: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/dacon/가중치/5-fold-v2/submission_final_ensemble_v2.csv
